In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from PidMLP import PidMLP

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
action_space_n = 50
action_bins = np.linspace(-2, 2, action_space_n)  # Define bins for discretization

In [3]:
# Load data
df = pd.read_csv('pid_controller_data.csv')

# Seperate features and labels
features = df[['target_lataccel', 'current_lataccel', 'vEgo', 'aEgo', 'roll', 'error', 'prev_error', 'integral', 'derivative', 'prev_derivative', "prev_action"]].values
labels = df['steerCommand'].values

# Discretize labels and one-hot encode
bin_indices = np.digitize(labels, action_bins, right=True) - 1
bin_indices = np.clip(bin_indices, 0, action_space_n - 1)

# Convert to tensors
features_tensor = torch.tensor(features, dtype=torch.float32, device=device)
labels_tensor = torch.tensor(bin_indices, dtype=torch.long, device=device)

# Create a TensorDataset
dataset = TensorDataset(features_tensor, labels_tensor)

# Create a DataLoader
dataloader = DataLoader(dataset, batch_size=8000, shuffle=True,)

In [4]:
len(dataloader)

623

In [5]:
# Instantiate model
model = PidMLP(in_features=11, output_space_n=action_space_n, dropout_rate=0.3).to(device)

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
schedular = optim.lr_scheduler.StepLR(optimizer, 5, 0.5)

# Training loop
epochs = 30
losses = []
for epoch in range(epochs):
    total_loss = 0
    correct_predictions = 0
    total_predictions = 0
    for inputs, targets in dataloader:
        inputs, targets = inputs.to(device), targets.to(device)
        outputs = model(inputs)
        
        # Calculate loss
        loss = criterion(outputs, targets)
        total_loss += loss.item()
        
        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # Calculate accuracy
        _, predicted = torch.max(outputs.data, 1)
        total_predictions += targets.size(0)
        correct_predictions += (predicted == targets).sum().item()

    schedular.step()

    # Compute average loss
    average_loss = total_loss / len(dataloader)
    losses.append(average_loss)

    # Compute accuracy
    accuracy = (correct_predictions / total_predictions) * 100
    
    # Print loss and accuracy
    print(f'End of Epoch {epoch+1}, Loss: {average_loss:.4f}, Accuracy: {accuracy:.2f}%')

End of Epoch 1, Loss: 0.8167, Accuracy: 69.29%
End of Epoch 2, Loss: 0.3226, Accuracy: 87.52%
End of Epoch 3, Loss: 0.2302, Accuracy: 91.21%
End of Epoch 4, Loss: 0.1819, Accuracy: 93.12%
End of Epoch 5, Loss: 0.1555, Accuracy: 94.16%
End of Epoch 6, Loss: 0.1336, Accuracy: 94.99%
End of Epoch 7, Loss: 0.1248, Accuracy: 95.33%
End of Epoch 8, Loss: 0.1174, Accuracy: 95.60%
End of Epoch 9, Loss: 0.1099, Accuracy: 95.90%
End of Epoch 10, Loss: 0.1042, Accuracy: 96.12%
End of Epoch 11, Loss: 0.0950, Accuracy: 96.46%
End of Epoch 12, Loss: 0.0922, Accuracy: 96.56%
End of Epoch 13, Loss: 0.0900, Accuracy: 96.66%
End of Epoch 14, Loss: 0.0872, Accuracy: 96.76%
End of Epoch 15, Loss: 0.0848, Accuracy: 96.85%
End of Epoch 16, Loss: 0.0801, Accuracy: 97.02%
End of Epoch 17, Loss: 0.0789, Accuracy: 97.07%
End of Epoch 18, Loss: 0.0776, Accuracy: 97.11%
End of Epoch 19, Loss: 0.0764, Accuracy: 97.14%
End of Epoch 20, Loss: 0.0753, Accuracy: 97.20%
End of Epoch 21, Loss: 0.0729, Accuracy: 97.29%
E

In [3]:
model = torch.load('./models/mlp_pid.pth')

In [6]:
torch.save(model, './models/mlp_pid.pth')

In [7]:
model.eval()
action = model(torch.tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], device=device, dtype=torch.float32))
print(action)

tensor([-16.0199, -16.3238, -17.6027, -18.3604, -21.4936, -21.3300, -19.3815,
        -20.1235, -20.5162, -19.2503, -20.1939, -18.9141, -18.1466, -17.4622,
        -15.0235, -14.5614, -13.9409, -15.3467, -22.3256, -39.0281, -43.6024,
        -25.6002, -22.9151, -20.4775, -17.5888, -32.0049,  -3.1843,   6.7437,
         12.2386,   9.3444,   3.9349,  -1.4775,  -7.0687, -12.1981, -17.4007,
        -23.2143, -28.8574, -33.3320, -39.4203, -43.8496, -45.7147, -49.4372,
        -51.0412, -48.7854, -46.4492, -44.4834, -34.7464, -16.1092, -15.4837,
        -15.1188], device='cuda:0', grad_fn=<ViewBackward0>)
